In [1]:
import urllib.request

for month in ["01", "02", "03"]:
    url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-{month}.parquet"
    print(f"Downloading 2024-{month}...")
    urllib.request.urlretrieve(url, f"yellow_tripdata_2024-{month}.parquet")
    print(f"Done!")

Done!
Done!
Done!


In [35]:
import pyarrow as pa
import pyarrow.csv as pcsv
import pyarrow.parquet as pq
import pandas as pd
import os
import time
from pathlib import Path

files = [
    "yellow_tripdata_2024-01.parquet",
    "yellow_tripdata_2024-02.parquet",
    "yellow_tripdata_2024-03.parquet"
]

# 1. Getting to know the data

How many rows does each monthly file have? How many in total?

In [3]:
summary = []
total_rows = 0

for file in files:
    table = pq.read_table(file)
    n_rows = table.num_rows
    n_cols = table.num_columns
    
    summary.append({
        "file": file,
        "rows": n_rows,
    })
    
    total_rows += n_rows

summary_df = pd.DataFrame(summary)
summary_df

,file,rows
0,yellow_tripdata_2024-01.parquet,2964624
1,yellow_tripdata_2024-02.parquet,3007526
2,yellow_tripdata_2024-03.parquet,3582628


In [4]:
print(f"Total rows: {total_rows:,}")

Total rows: 9,554,778


What columns are available? What are their data types?

In [5]:
table_01 = pq.read_table(files[0])
schema_df = pd.DataFrame({
    "column": table_01.schema.names,
    "dtype": [str(field.type) for field in table_01.schema]
})

schema_df

,column,dtype
0,VendorID,int32
1,tpep_pickup_datetime,timestamp[us]
2,tpep_dropoff_datetime,timestamp[us]
3,passenger_count,int64
4,trip_distance,double
5,RatecodeID,int64
6,store_and_fwd_flag,large_string
7,PULocationID,int32
8,DOLocationID,int32
9,payment_type,int64


Exploring rows, to understand the data

In [6]:
sample_df = pd.read_parquet(files[0]).head(10)
sample_df

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,2,2024-01-01 00:57:55,2024-01-01 01:17:43,1.0,1.72,1.0,N,186,79,2,17.7,1.0,0.5,0.00,0.0,1.0,22.70,2.5,0.00
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1.0,1.80,1.0,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.00
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1.0,4.70,1.0,N,236,79,1,23.3,3.5,0.5,3.00,0.0,1.0,31.30,2.5,0.00
3,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1.0,1.40,1.0,N,79,211,1,10.0,3.5,0.5,2.00,0.0,1.0,17.00,2.5,0.00
4,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1.0,0.80,1.0,N,211,148,1,7.9,3.5,0.5,3.20,0.0,1.0,16.10,2.5,0.00
5,1,2024-01-01 00:54:08,2024-01-01 01:26:31,1.0,4.70,1.0,N,148,141,1,29.6,3.5,0.5,6.90,0.0,1.0,41.50,2.5,0.00
6,2,2024-01-01 00:49:44,2024-01-01 01:15:47,2.0,10.82,1.0,N,138,181,1,45.7,6.0,0.5,10.00,0.0,1.0,64.95,0.0,1.75
7,1,2024-01-01 00:30:40,2024-01-01 00:58:40,0.0,3.00,1.0,N,246,231,2,25.4,3.5,0.5,0.00,0.0,1.0,30.40,2.5,0.00
8,2,2024-01-01 00:26:01,2024-01-01 00:54:12,1.0,5.44,1.0,N,161,261,2,31.0,1.0,0.5,0.00,0.0,1.0,36.00,2.5,0.00
9,2,2024-01-01 00:28:08,2024-01-01 00:29:16,1.0,0.04,1.0,N,113,113,2,3.0,1.0,0.5,0.00,0.0,1.0,8.00,2.5,0.00


# 2. MongoDB — Loading and Querying Semi-structured Data

## 2.1. Load data into MongoDB

Export a sample of 100,000 rows from the Parquet data as JSON Lines, and import it into MongoDB using `mongoimport`

In [7]:
# Export 100,000 rows to JSON Lines for MongoDB import

table_sample = pq.read_table(files[0]).slice(0, 100_000)
df_sample = table_sample.to_pandas()

df_sample.to_json("trips_sample.json", orient="records", lines=True)

print(f"Exported {len(df_sample):,} rows to trips_sample.json")

Exported 100,000 rows to trips_sample.json


In [22]:
# Copy file into MongoDB container
!docker cp trips_sample.json mongodb:/trips_sample.json
!docker exec mongodb ls -l /trips_sample.json
!docker exec mongodb mongoimport --db taxi --collection trips --drop --file /trips_sample.json

-rwxr-xr-x 1 root root 39442664 Mar 15 13:59 /trips_sample.json


2026-03-15T14:41:12.070+0000	connected to: mongodb://localhost/
2026-03-15T14:41:12.071+0000	dropping: taxi.trips
2026-03-15T14:41:15.071+0000	[########................] taxi.trips	12.9MB/37.6MB (34.4%)
2026-03-15T14:41:18.072+0000	[###############.........] taxi.trips	24.5MB/37.6MB (65.2%)
2026-03-15T14:41:20.592+0000	[########################] taxi.trips	37.6MB/37.6MB (100.0%)
2026-03-15T14:41:20.593+0000	100000 document(s) imported successfully. 0 document(s) failed to import.


## 2.2 Explore and query

How many documents are in the collection?

In [15]:
!docker exec mongodb mongosh taxi --eval 'db.trips.countDocuments()'

100000


Find all trips with more than 4 passengers

In [18]:
!docker exec mongodb mongosh taxi --eval "db.trips.find({ passenger_count: { $gt: 4 } }).limit(5)"

[
  {
    _id: ObjectId('69b6bf1b823b288ae3fd16b3'),
    VendorID: 2,
    tpep_pickup_datetime: 1704068526,
    tpep_dropoff_datetime: 1704070288,
    passenger_count: 5,
    trip_distance: 3.1,
    RatecodeID: 1,
    store_and_fwd_flag: 'N',
    PULocationID: 143,
    DOLocationID: 170,
    payment_type: 1,
    fare_amount: 26.8,
    extra: 1,
    mta_tax: 0.5,
    tip_amount: 6.36,
    tolls_amount: 0,
    improvement_surcharge: 1,
    total_amount: 38.16,
    congestion_surcharge: 2.5,
    Airport_fee: 0
  },
  {
    _id: ObjectId('69b6bf1b823b288ae3fd16b9'),
    VendorID: 2,
    tpep_pickup_datetime: 1704068092,
    tpep_dropoff_datetime: 1704068258,
    passenger_count: 5,
    trip_distance: 0.55,
    RatecodeID: 1,
    store_and_fwd_flag: 'N',
    PULocationID: 239,
    DOLocationID: 143,
    payment_type: 1,
    fare_amount: 5.1,
    extra: 1,
    mta_tax: 0.5,
    tip_amount: 1.52,
    tolls_amount: 0,
    improvement_surcharge: 1,
    total_amount: 11.62,
    congestion_surcha

Find the average `tip_amount` grouped by `payment_type`.

In [19]:
!docker exec mongodb mongosh taxi --eval "db.trips.aggregate([{ $group: { _id: '$payment_type', avg_tip_amount: { $avg: '$tip_amount' } } }])"

[
  { _id: 4, avg_tip_amount: 0.055960544856740256 },
  { _id: 3, avg_tip_amount: 0 },
  { _id: 2, avg_tip_amount: 0.001203177139634347 },
  { _id: 1, avg_tip_amount: 4.705123567618947 }
]


Find the top 5 longest trips by `trip_distance`

In [21]:
!docker exec mongodb mongosh taxi --eval "db.trips.find().sort({ trip_distance: -1 })"

[
  {
    _id: ObjectId('69b6bf23823b288ae3fe4f62'),
    VendorID: 1,
    tpep_pickup_datetime: 1704183198,
    tpep_dropoff_datetime: 1704188610,
    passenger_count: 1,
    trip_distance: 971.8,
    RatecodeID: 99,
    store_and_fwd_flag: 'N',
    PULocationID: 237,
    DOLocationID: 265,
    payment_type: 1,
    fare_amount: 21.5,
    extra: 0,
    mta_tax: 0.5,
    tip_amount: 0,
    tolls_amount: 0,
    improvement_surcharge: 1,
    total_amount: 23,
    congestion_surcharge: 0,
    Airport_fee: 0
  },
  {
    _id: ObjectId('69b6bf23823b288ae3fe4f5e'),
    VendorID: 1,
    tpep_pickup_datetime: 1704184841,
    tpep_dropoff_datetime: 1704185727,
    passenger_count: 1,
    trip_distance: 964.6,
    RatecodeID: 99,
    store_and_fwd_flag: 'N',
    PULocationID: 71,
    DOLocationID: 265,
    payment_type: 1,
    fare_amount: 39.5,
    extra: 0,
    mta_tax: 0.5,
    tip_amount: 0,
    tolls_amount: 0,
    improvement_surcharge: 1,
    total_amount: 41,
    congestion_surcharge: 0,
 

## 2.3 Reflection
MongoDB is less suitable here because the workload is analytical. Analytical workloads often scan large amounts of data, usually need only a few columns, and benefit from columnar formats. MongoDB is not ideal for analytics because it does not use columnar storage.

Parquet + DuckDB is a better fit because Parquet is columnar, compressed by default (different compression algorithms available), and supports statistics. This reduces I/O and makes analytical queries more efficient.

# 3. Format Comparison

## 3.1. Read and combine the data

In [28]:
tables = [pq.read_table(file) for file in files]
combined_table = pa.concat_tables(tables)

print("Rows:", combined_table.num_rows)
print("Columns:", combined_table.num_columns)

Rows: 9554778
Columns: 19


## 3.2 Export to other formats

In [29]:
#CSV export (full dataset)
combined_df = combined_table.to_pandas()
combined_df.to_csv("trips_full.csv", index=False)

#JSON Lines export (sample of 100,000 rows)
table_sample = combined_table.slice(0, 100_000)
df_sample = table_sample.to_pandas()

df_sample.to_json("trips_sample_100k.json", orient="records", lines=True)



## 3.3 Compare file sizes

In [32]:
parquet_size = 0

for file in files:
    parquet_size += os.path.getsize(file)

print(f"Parquet original files: {parquet_size / 1e6:.1f} MB")

for name, path in [("CSV", "trips_full.csv"), ("JSON (100k)", "trips_sample_100k.json")]:
    size = os.path.getsize(path)
    print(f"{name}: {size / 1e6:.1f} MB")

Parquet original files: 160.4 MB
CSV: 1006.9 MB
JSON (100k): 39.4 MB


In [57]:
csv_size = os.path.getsize("trips_full.csv")
json_sample_size = os.path.getsize("trips_sample_100k.json")

json_extrapolated_size = json_sample_size * (combined_table.num_rows / 100_000)

size_table = pd.DataFrame([
    {
        "Format": "Parquet original files",
        "File Size (MB)": round(parquet_size / 1e6, 1),
        "Ratio vs Parquet": "1.0x"
    },
    {
        "Format": "CSV",
        "File Size (MB)": round(csv_size / 1e6, 1),
        "Ratio vs Parquet": f"{csv_size / parquet_size:.2f}x"
    },
    {
        "Format": "JSON lines (100K rows, extrapolated total)",
        "File Size (MB)": round(json_extrapolated_size / 1e6, 1),
        "Ratio vs Parquet": f"{json_extrapolated_size / parquet_size:.2f}x"
    }
])

size_table

,Format,File Size (MB),Ratio vs Parquet
0,Parquet original files,160.4,1.0x
1,CSV,1006.9,6.28x
2,"JSON lines (100K rows, extrapolated total)",3768.7,23.50x


## 3.4 Compare read times

In [36]:
start = time.time()

tables = [pq.read_table(file) for file in files]
parquet_table = pa.concat_tables(tables)

parquet_read_time = time.time() - start
print(f"Parquet read time: {parquet_read_time:.2f}s")

start = time.time()

csv_table = pcsv.read_csv("trips_full.csv")

csv_read_time = time.time() - start
print(f"CSV read time: {csv_read_time:.2f}s")

Parquet read time: 0.87s
CSV read time: 12.02s


## 3.5 Compression comparison

In [37]:
compression_results = []

for codec in ["snappy", "zstd", "none", "gzip"]:
    start = time.time()
    pq.write_table(combined_table, f"trips_{codec}.parquet", compression=codec)
    write_time = time.time() - start
    size = os.path.getsize(f"trips_{codec}.parquet")
    
    compression_results.append({
        "compression": codec,
        "file_size_mb": size / 1e6,
        "write_time_s": write_time
    })
    
    print(f"{codec:8s}: {size / 1e6:8.1f} MB  write: {write_time:.2f}s")

snappy  :    195.4 MB  write: 27.52s
zstd    :    158.4 MB  write: 14.59s
none    :    249.9 MB  write: 10.01s
gzip    :    149.9 MB  write: 146.28s


In [38]:
for result in compression_results:
    codec = result["compression"]
    
    start = time.time()
    t = pq.read_table(f"trips_{codec}.parquet")
    read_time = time.time() - start
    
    result["read_time_s"] = read_time
    
    print(f"{codec:8s}: read {read_time:.2f}s")

snappy  : read 0.71s
zstd    : read 1.60s
none    : read 1.32s
gzip    : read 3.17s


In [39]:
compression_df = pd.DataFrame(compression_results)
compression_df

,compression,file_size_mb,write_time_s,read_time_s
0,snappy,195.410094,27.519080,0.714159
1,zstd,158.409287,14.588077,1.602342
2,none,249.866341,10.007856,1.318115
3,gzip,149.887799,146.280546,3.171536


Gzip produced the smallest file, but it was much slower to write and read. Snappy and Zstd gave a better balance between compression and performance, so I would recommend one of these.

## 3.6 Column pruning

In [40]:
start = time.time()
t_all = pq.read_table("trips_snappy.parquet")
time_all = time.time() - start

start = time.time()
t_pruned = pq.read_table("trips_snappy.parquet",
                         columns=["payment_type", "tip_amount"])
time_pruned = time.time() - start

print(f"All columns: {time_all:.2f}s")
print(f"2 columns:   {time_pruned:.2f}s")

All columns: 0.94s
2 columns:   0.16s


Ruffly 6 times faster.

# 4. Partitioning Strategy 

## 4.1 Choose a partition column

In [41]:
import pyarrow.compute as pc

months = pc.month(combined_table.column("tpep_pickup_datetime"))
table_with_month = combined_table.append_column("month", months)

pq.write_to_dataset(table_with_month, "trips_partitioned",
                    partition_cols=["month"])

## 4.2 Inspect the partitioned output

In [42]:
# Number of partitions

partitions = os.listdir("trips_partitioned")

print("Number of partitions:", len(partitions))
print("Partitions:", partitions)

Number of partitions: 5
Partitions: ['month=1', 'month=12', 'month=2', 'month=3', 'month=4']


In [43]:
# File size in each partition
partition_sizes = []

for partition in partitions:
    path = os.path.join("trips_partitioned", partition)
    
    size = 0
    for file in os.listdir(path):
        file_path = os.path.join(path, file)
        size += os.path.getsize(file_path)
    
    size_mb = size / 1e6
    partition_sizes.append((partition, size_mb))
    
    print(f"{partition}: {size_mb:.1f} MB")

month=1: 62.4 MB
month=12: 0.0 MB
month=2: 62.7 MB
month=3: 75.2 MB
month=4: 0.0 MB


The partitions are not fully balanced. The main partitions for months 1, 2, and 3 are of similar size, but months 4 and 12 are almost empty.

## 4.3 Partition pruning

In [45]:
import pyarrow.dataset as ds

# With partition pruning (month=1)
start = time.time()
dataset = ds.dataset("trips_partitioned", partitioning="hive")
t_pruned = dataset.to_table(filter=ds.field("month") == 1)
time_pruned = time.time() - start

# Without pruning 
start = time.time()
t_all = pq.read_table("trips_snappy.parquet")
t_filtered = t_all.filter(pc.equal(pc.month(t_all.column("tpep_pickup_datetime")), 1))
time_full = time.time() - start

print(f"With partition pruning: {time_pruned:.2f}s ({t_pruned.num_rows:,} rows)")
print(f"Without pruning:        {time_full:.2f}s ({t_filtered.num_rows:,} rows)")
print(f"Speedup: {time_full / time_pruned:.2f}x")

With partition pruning: 0.20s (2,964,621 rows)
Without pruning:        2.10s (2,964,621 rows)
Speedup: 10.69x


## 4.4 Over-partitioning

In [46]:
years = pc.year(combined_table.column("tpep_pickup_datetime"))
months = pc.month(combined_table.column("tpep_pickup_datetime"))
days = pc.day(combined_table.column("tpep_pickup_datetime"))

table_with_ymd = combined_table.append_column("year", years)
table_with_ymd = table_with_ymd.append_column("month", months)
table_with_ymd = table_with_ymd.append_column("day", days)

pq.write_to_dataset(table_with_ymd, "trips_partitioned_by_day",
                    partition_cols=["year", "month", "day"])

In [49]:
partition_paths = []

for root, dirs, files in os.walk("trips_partitioned_by_day"):
    if files:
        partition_paths.append(root)

print("partition directories:", len(partition_paths))
partition_paths[:10]

partition directories: 96


['trips_partitioned_by_day\\year=2002\\month=12\\day=31',
 'trips_partitioned_by_day\\year=2008\\month=12\\day=31',
 'trips_partitioned_by_day\\year=2009\\month=1\\day=1',
 'trips_partitioned_by_day\\year=2023\\month=12\\day=31',
 'trips_partitioned_by_day\\year=2024\\month=1\\day=1',
 'trips_partitioned_by_day\\year=2024\\month=1\\day=10',
 'trips_partitioned_by_day\\year=2024\\month=1\\day=11',
 'trips_partitioned_by_day\\year=2024\\month=1\\day=12',
 'trips_partitioned_by_day\\year=2024\\month=1\\day=13',
 'trips_partitioned_by_day\\year=2024\\month=1\\day=14']

In [48]:
partition_sizes = []

for path in partition_paths:
    size = 0
    for file in os.listdir(path):
        file_path = os.path.join(path, file)
        size += os.path.getsize(file_path)
    
    partition_sizes.append({
        "partition": path,
        "size_mb": size / 1e6
    })

partition_sizes_df = pd.DataFrame(partition_sizes)
partition_sizes_df

,partition,size_mb
0,trips_partitioned_by_day\year=2002\month=12\da...,0.013766
1,trips_partitioned_by_day\year=2008\month=12\da...,0.005992
2,trips_partitioned_by_day\year=2009\month=1\day=1,0.017593
3,trips_partitioned_by_day\year=2023\month=12\da...,0.006513
4,trips_partitioned_by_day\year=2024\month=1\day=1,1.854848
...,...,...
91,trips_partitioned_by_day\year=2024\month=3\day=6,2.756333
92,trips_partitioned_by_day\year=2024\month=3\day=7,2.543334
93,trips_partitioned_by_day\year=2024\month=3\day=8,2.543758
94,trips_partitioned_by_day\year=2024\month=3\day=9,2.880230


This is not a good partitioning strategy. Partitioning by day creates many small partitions (96 directories in this case), and many of them contain only very small files. This leads to the small files problem: metadata overhead increases, compression becomes less efficient, and reads can become slower.

## 4.5 Metadata inspection

In [53]:
metadata = pq.read_metadata("trips_snappy.parquet")
print(metadata)

  created_by: parquet-cpp-arrow version 23.0.1
  num_columns: 19
  num_rows: 9554778
  num_row_groups: 10
  format_version: 2.6
  serialized_size: 25787


In [54]:
print("Number of row groups:", metadata.num_row_groups)

Number of row groups: 10


In [55]:
print(metadata.schema)

required group field_id=-1 schema {
  optional int32 field_id=-1 VendorID;
  optional int64 field_id=-1 tpep_pickup_datetime (Timestamp(isAdjustedToUTC=false, timeUnit=microseconds, is_from_converted_type=false, force_set_converted_type=false));
  optional int64 field_id=-1 tpep_dropoff_datetime (Timestamp(isAdjustedToUTC=false, timeUnit=microseconds, is_from_converted_type=false, force_set_converted_type=false));
  optional int64 field_id=-1 passenger_count;
  optional double field_id=-1 trip_distance;
  optional int64 field_id=-1 RatecodeID;
  optional binary field_id=-1 store_and_fwd_flag (String);
  optional int32 field_id=-1 PULocationID;
  optional int32 field_id=-1 DOLocationID;
  optional int64 field_id=-1 payment_type;
  optional double field_id=-1 fare_amount;
  optional double field_id=-1 extra;
  optional double field_id=-1 mta_tax;
  optional double field_id=-1 tip_amount;
  optional double field_id=-1 tolls_amount;
  optional double field_id=-1 improvement_surcharge;
  op

In [56]:
column_name = "trip_distance"

for i in range(metadata.num_row_groups):
    row_group = metadata.row_group(i)
    
    for j in range(row_group.num_columns):
        column = row_group.column(j)
        
        if column.path_in_schema == column_name:
            stats = column.statistics
            print(f"Row group {i}: min = {stats.min}, max = {stats.max}")

Row group 0: min = -0.0, max = 10879.28
Row group 1: min = -0.0, max = 15400.32
Row group 2: min = -0.0, max = 312722.3
Row group 3: min = -0.0, max = 98229.4
Row group 4: min = -0.0, max = 63515.2
Row group 5: min = -0.0, max = 222478.29
Row group 6: min = -0.0, max = 66907.9
Row group 7: min = -0.0, max = 61099.8
Row group 8: min = -0.0, max = 176836.3
Row group 9: min = -0.0, max = 176744.79


Parquet stores min/max statistics for each row group. This can be used for predicate pushdown. If the filter condition cannot be satisfied in a row group, that row group can be skipped entirely.